In [ ]:
## ----------Typical response API - OpenAI

response = client.responses.create(
    model = MODEL, 
    input = prompt, 
    temperature = 0.7    # lower for more deterministic output, higher means more predictibility
)

In [ ]:
##------------Structured output ---using manual JSON format for output
import json

#response = client.responses.create(
response = ollama.responses.create(
    model = MODEL,
    input = [{"role":"developer", "content" : "You are a User Input form generator AI. Convert the user input into a UI form"},
             {"role":"user", "content":"Build a admission form for preschool"}],
    text={
        "admission_schema" : {
            "type": "json_schema",
            "name": "preschool_admission",
            "schema": {
                "type": "object",
                "properties": {
                    "child_name": {
                        "type": "string",
                        "description": "Full name of the child"
                    },
                    "date_of_birth": {
                        "type": "string",
                        "description": "Date of birth of the child"
                    },
                    "parent_name": {
                        "type": "string",
                        "description": "Full name of the parent or guardian"
                    },
                    "parent_phone": {
                        "type": "string",
                        "description": "Parent or guardian phone number"
                    },
                    "admission_class": {
                        "type": "string",
                        "description": "Class the child is applying for, such as Nursery, LKG, or UKG"
                    }
                },
                "required": [
                    "child_name",
                    "date_of_birth",
                    "parent_name",
                    "parent_phone",
                    "admission_class"
                ],
                "additionalProperties": False
            },
            "strict": True,
            },
        },
)
print("RAW OUTPUT:")
print(response.output_text)

ui = json.loads(response.output_text)
print(ui)

RAW OUTPUT:
Here's a sample admission form for a preschool program at Orion:

**Orion Preschool Admission Form**

**Preschool Program Information:**

* Name of the child: _____________________
* Parent/Guardian Name: _______________________________________
* Address: _______________________________________________________
* City, State, Zip: _______________________________________________

**Parent/Guardian Information:**

* Contact Phone Number: _______________________________________
* Email Address: _______________________________________________
* Days and Times to be available for pickup/ drop-off: _____________________

**Child's Information:**

* Date of Birth (mm/dd/yyyy): ____________________________________
* Grade Level in the upcoming school year: __________________________________
* Previous preschool experience (if any): Yes No

**Medical Information:**

* Does your child have any known medical conditions? (e.g. diabetes, food allergies) Yes No
* If yes, please provide a 

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [16]:
## Structured output using Pydantic and basemodel 

from pydantic import BaseModel

class Step(BaseModel):
    explanation : str
    output : str

class MathReason(BaseModel):
    steps : list[Step]
    final_answer : str

response = ollama.responses.parse(
    model = MODEL, 
    input = [{"role": "developer", "content" : "You are math teacher. Build solution step by step for user's question"},
             {"role" : "user", "content" : "Can you solve 100 * (5+20) = ?"},],
    text_format = MathReason,
)

solution = response.output_parsed
solution.steps

[Step(explanation="First, we need to evaluate the expression inside the parentheses.  In this case, it is '5+20'. This equals 25.", output='result=100*25'),
 Step(explanation='Next, multiply 100 by the result from step 1: result = 100 * 25', output='result=2500')]

In [ ]:
## Image generation 
import base64
from IPython.display import display, Image as IPImage

# Generate an image using the Responses API with the image_generation tool
response = client.responses.create(
    model=MODEL,
    input="Generate a watercolor painting of a cozy coffee shop on a rainy day",
    tools=[{
        "type": "image_generation",
        "quality": "high",
        "size": "1024x1024"
    }]
)

# Extract the base64-encoded image from the response output
image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

if image_data:
    image_bytes = base64.b64decode(image_data[0])

    # Save to file
    with open("generated_image.png", "wb") as f:
        f.write(image_bytes)

    # Display inline in the notebook
    display(IPImage(data=image_bytes))
    print("Image generated and saved to generated_image.png")

In [ ]:
# You can also generate images with transparent backgrounds (useful for logos, sprites, etc.)
response = client.responses.create(
    model=MODEL,
    input="A cute robot mascot giving a thumbs up, simple flat design",
    tools=[{
        "type": "image_generation",
        "background": "transparent",
        "quality": "high",
        "output_format": "png"
    }]
)

image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

if image_data:
    image_bytes = base64.b64decode(image_data[0])

    with open("mascot_transparent.png", "wb") as f:
        f.write(image_bytes)

# Add at top of cell: from IPython.display import display
    display(IPImage(data=image_bytes))
    print("Transparent background image generated!")

In [ ]:
#  Multimodal Capabilities - Vision
# For notebook demonstration, we'll use a placeholder URL
image_url = "https://images.unsplash.com/photo-1579546929518-9e396f3cc809?ixlib=rb-4.0.3&ixid=MnwxMjA3fDB8MHxleHBsb3JlLWZlZWR8MXx8fGVufDB8fHx8&w=1000&q=80"

response = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "what's in this image?"},
            {
                "type": "input_image",
                "image_url": image_url,
            },
        ],
    }],
)

print(response.output_text)

In [ ]:
# Text to Speech 
speech_file_path = "speech.mp3"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Today is a wonderful day to build something people love!",
    instructions="Speak in a cheerful and positive tone.",
) as response:
    response.stream_to_file(speech_file_path)

In [ ]:
#  Audio Capabilities - Speech to Text
audio_file= open("speech.mp3", "rb")

transcription = client.audio.transcriptions.create(
    model="gpt-4o-mini-transcribe", file=audio_file
)

print(transcription.text)
audio_file.close()

In [ ]:
#-------Streaming-------------
# stream responses from OpenAI models to display output incrementally as it is generated.
# Request the model to recite the tongue twister five times using streaming
stream = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": "Recite 'Peter Piper picked a peck of pickled peppers' five times in a row."
    }],
    stream=True                  # <--------------------------
)

# Iterate over streaming events and print details
for event in stream:
    if hasattr(event, 'type'):                       # Depending on the event type, you can handle it accordingly
        print(f"Event Type: {event.type}")
    if hasattr(event, 'delta') and event.delta:      # If the event includes a delta with content, print it
        print(f"Delta Content: {event.delta}")


In [43]:
###------------ Tool calling----------
# Structure - Define logic function, Define structure parameters for tools, call tool from response API, 
# call Tool and parse output parameters and then use it for further processing

import requests
from llm_config import MODEL, ollama    

#-----Define logic function 
def get_weather_info(latitude, longitude):
    response = requests.get( 
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}"
        f"&current=temperature_2m,wind_speed_10m" )  
    data = response.json()
    return (
        data["current"]["temperature_2m"],
        data["current"]["wind_speed_10m"]
    )
temperature, wind_speed = get_weather_info(25, 35) 
#print(temperature, wind_speed)

#-----Defune structure parameters for tools ---
tools = [{
    "type" : "function",
    "name" : "get_weather_info",
    "description" : "Get the weather info",
    "parameters" : {
        "type"       : "object",
        "properties" : {
            "latitude"  : {"type" : "number"},
            "longitude" : {"type" : "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

input_msg = [{"role" : "user", "content" : "What is weather in Pune today?"}]
#--------------Response API call
response = ollama.chat.completions.create(
    model = MODEL,
    messages = input_msg,
    tools = tools
)

#------Call tool and parse -----
tool_call = response.choices[0].message.tool_calls[0]
print(tool_call)
args = json.loads(tool_call.function.arguments)
print(args)
result = get_weather_info(args["latitude"], args["longitude"])
print(result)

ChatCompletionMessageFunctionToolCall(id='call_2qam957v', function=Function(arguments='{"city":"Pune"}', name=''), type='function', index=0)
{'city': 'Pune'}


KeyError: 'latitude'

In [ ]:
#-------------Reasoning Models 

response = client.responses.create(
    model = MODEL, 
    reasoning = {"effort" : "high"}     # high or Xhigh means model will think more --will cost more
    text= {"verbosity" : "high"}        # more context will be giving in output ... will cost more
    input = "Explain how to travel arcoss Europe"
)